# Materialized Tables: Neo4j Graph Data as Delta Tables

Materializes Neo4j graph data as **managed Delta tables** in Unity Catalog, making it
queryable alongside lakehouse tables with standard SQL. Once materialized, GROUP BY,
ORDER BY, WHERE, JOINs, and aggregations all work seamlessly — no JDBC at query time.

**Why materialize?**
- **Genie / AI agents** can query graph data via natural language → SQL
- **Full SQL support** — no limitations from real-time JDBC translation
- **Performance** — Delta tables are optimized for analytical queries
- **Governance** — materialized tables are visible in Catalog Explorer with full lineage

**Prerequisites:**
- Run `01-simple-connect-test.ipynb` to validate connectivity and create the UC JDBC connection
- Run `02-federated-queries.ipynb` to load graph data into Neo4j and Delta tables into the lakehouse

## Architecture

```
┌──────────────────────────────────────────────────────────────────────┐
│                  Genie / SQL Tool / Agent                           │
│           (natural language or SQL queries)                         │
└─────────────────────────────┬────────────────────────────────────────┘
                              │
                              ▼
┌──────────────────────────────────────────────────────────────────────┐
│                      Spark SQL Engine                               │
│                                                                      │
│   Query hits Delta tables only — no JDBC calls at query time        │
│                                                                      │
│   ┌─────────────────────┐    ┌─────────────────────┐                │
│   │   Delta Tables       │    │  Materialized from   │                │
│   │   (from notebook 02) │    │  Neo4j (this notebook)│                │
│   │                      │    │                      │                │
│   │ • financial_metrics  │    │ • neo4j_companies    │                │
│   │ • asset_managers     │    │ • neo4j_products     │                │
│   │ • asset_mgr_holdings │    │ • neo4j_risk_factors │                │
│   │ • companies          │    │ • neo4j_competitors  │                │
│   └─────────────────────┘    └──────────┬───────────┘                │
└──────────────────────────────────────────┼───────────────────────────┘
                              ▲            │ Materialization
                              │            │ (periodic refresh)
                              │            ▼
                    ┌─────────┴────────────────────────┐
                    │         Neo4j Aura               │
                    │    (Company, Product, RiskFactor  │
                    │     OFFERS, COMPETES_WITH,        │
                    │     HAS_RISK relationships)       │
                    └──────────────────────────────────┘
```

---

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these two sections only
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://123456.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
# If your catalog name contains hyphens (e.g., "uc-w-neo4j"), Spark SQL requires
# backtick quoting. Without it you'll get:
#   [INVALID_IDENTIFIER] The unquoted identifier uc-w-neo4j is invalid
#   and must be back quoted as: `uc-w-neo4j`.
# FQN handles this automatically.
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
UC_CONNECTION_NAME = f"`{UC_CATALOG}`.`{UC_SCHEMA}`.`neo4j_jdbc`"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Verify Data Sources

Confirm both the lakehouse tables (from notebook 02) and Neo4j graph data are accessible.

In [ ]:
# Verify Delta lakehouse tables from notebook 02
print("=" * 60)
print("DELTA LAKEHOUSE TABLES (from notebook 02)")
print("=" * 60)

for table in ["companies", "financial_metrics", "asset_managers", "asset_manager_holdings"]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.{table}").collect()[0]["cnt"]
    print(f"  {table}: {count} rows")

print("\nSample companies:")
spark.sql(f"SELECT companyId, name, ticker FROM {FQN}.companies ORDER BY companyId").show(truncate=False)

In [ ]:
# Verify Neo4j graph data via UC JDBC
print("=" * 60)
print("NEO4J KNOWLEDGE GRAPH (via UC JDBC)")
print("=" * 60)

neo4j_counts = {
    "Company": "SELECT COUNT(*) AS cnt FROM Company",
    "Product": "SELECT COUNT(*) AS cnt FROM Product",
    "RiskFactor": "SELECT COUNT(*) AS cnt FROM RiskFactor",
}

for label, query in neo4j_counts.items():
    result = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", query) \
        .option("customSchema", "cnt LONG") \
        .load().collect()
    print(f"  {label}: {result[0]['cnt']} nodes")

print("\nBoth data sources verified.")
print("Status: PASS")

---

## Section 2: Materialize Neo4j Data as Delta Tables

Each Neo4j node label is read via `dbtable` + `customSchema`, then written as a **managed
Delta table** in Unity Catalog. Relationship data (OFFERS, COMPETES_WITH, HAS_RISK) is
read via the Neo4j Python driver and written as mapping tables.

**Why `dbtable` + `customSchema`?** Two Neo4j JDBC behaviors require this approach:
1. The `query` option triggers Spark's subquery wrapping for schema inference, which
   the Neo4j SQL translator must clean up — `dbtable` avoids this entirely.
2. Neo4j JDBC returns `NullType` during schema inference — `customSchema` provides
   explicit types so Spark can read the data correctly.

**Why Python driver for relationships?** Relationship traversals (e.g., "which company
offers which product") require Cypher queries that return denormalized results. The Python
driver is the most direct way to run these and create Spark DataFrames.

Re-run this section to refresh data from Neo4j.

In [ ]:
# =============================================================================
# Materialize node labels via dbtable + customSchema
# =============================================================================
# Reads Company, Product, and RiskFactor nodes from Neo4j and writes them
# as managed Delta tables in Unity Catalog.

print("=" * 60)
print("MATERIALIZING: Node Labels (via UC JDBC dbtable)")
print("=" * 60)

node_tables = {
    "neo4j_companies": {
        "dbtable": "Company",
        "schema": "companyId STRING, name STRING, ticker STRING",
    },
    "neo4j_products": {
        "dbtable": "Product",
        "schema": "productId STRING, name STRING",
    },
    "neo4j_risk_factors": {
        "dbtable": "RiskFactor",
        "schema": "riskId STRING, name STRING",
    },
}

for table_name, config in node_tables.items():
    fqn = f"{FQN}.{table_name}"

    # Drop existing table
    spark.sql(f"DROP TABLE IF EXISTS {fqn}")

    # Read from Neo4j via dbtable + customSchema
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("dbtable", config["dbtable"]) \
        .option("customSchema", config["schema"]) \
        .load()

    # Write as managed Delta table
    df.write.format("delta").mode("overwrite").saveAsTable(fqn)

    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {fqn}").collect()[0]["cnt"]
    print(f"[OK] {fqn}: {count} rows (from Neo4j {config['dbtable']} nodes)")

print("\nStatus: PASS")

In [ ]:
# =============================================================================
# Materialize relationship data via Neo4j Python driver
# =============================================================================
# Reads OFFERS, COMPETES_WITH, and HAS_RISK relationships from Neo4j using
# Cypher, creates Spark DataFrames, and writes as managed Delta tables.

from neo4j import GraphDatabase
from pyspark.sql import Row

print("=" * 60)
print("MATERIALIZING: Relationships (via Python driver)")
print("=" * 60)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    with driver.session() as session:
        # --- neo4j_company_products: OFFERS relationships ---
        result = session.run("""
            MATCH (c:Company)-[:OFFERS]->(p:Product)
            RETURN c.companyId AS companyId, c.name AS companyName,
                   p.productId AS productId, p.name AS productName
            ORDER BY c.companyId, p.productId
        """)
        rows = [Row(**record.data()) for record in result]
        df = spark.createDataFrame(rows)

        fqn = f"{FQN}.neo4j_company_products"
        spark.sql(f"DROP TABLE IF EXISTS {fqn}")
        df.write.format("delta").mode("overwrite").saveAsTable(fqn)
        print(f"[OK] {fqn}: {len(rows)} rows (Company-[:OFFERS]->Product)")

        # --- neo4j_competitors: COMPETES_WITH relationships ---
        result = session.run("""
            MATCH (c:Company)-[:COMPETES_WITH]->(comp:Company)
            RETURN c.companyId AS companyId, c.name AS companyName,
                   comp.name AS competitorName, comp.companyId AS competitorId
            ORDER BY c.companyId, comp.name
        """)
        rows = [Row(**record.data()) for record in result]
        df = spark.createDataFrame(rows)

        fqn = f"{FQN}.neo4j_competitors"
        spark.sql(f"DROP TABLE IF EXISTS {fqn}")
        df.write.format("delta").mode("overwrite").saveAsTable(fqn)
        print(f"[OK] {fqn}: {len(rows)} rows (Company-[:COMPETES_WITH]->Company)")

        # --- neo4j_company_risks: HAS_RISK relationships ---
        result = session.run("""
            MATCH (c:Company)-[:HAS_RISK]->(r:RiskFactor)
            RETURN c.companyId AS companyId, c.name AS companyName,
                   r.riskId AS riskId, r.name AS riskName
            ORDER BY c.companyId, r.riskId
        """)
        rows = [Row(**record.data()) for record in result]
        df = spark.createDataFrame(rows)

        fqn = f"{FQN}.neo4j_company_risks"
        spark.sql(f"DROP TABLE IF EXISTS {fqn}")
        df.write.format("delta").mode("overwrite").saveAsTable(fqn)
        print(f"[OK] {fqn}: {len(rows)} rows (Company-[:HAS_RISK]->RiskFactor)")

    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")
finally:
    driver.close()

In [ ]:
# Verify all materialized tables
print("=" * 60)
print("ALL MATERIALIZED NEO4J TABLES")
print("=" * 60)
print(f"Schema: {FQN}\n")

neo4j_tables = {
    "neo4j_companies": "Company nodes (dbtable)",
    "neo4j_products": "Product nodes (dbtable)",
    "neo4j_risk_factors": "RiskFactor nodes (dbtable)",
    "neo4j_company_products": "OFFERS relationships (Python driver)",
    "neo4j_competitors": "COMPETES_WITH relationships (Python driver)",
    "neo4j_company_risks": "HAS_RISK relationships (Python driver)",
}

for table_name, description in neo4j_tables.items():
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.{table_name}").collect()[0]["cnt"]
    print(f"  {table_name}: {count} rows — {description}")

print("\nAll Neo4j graph data is now queryable as Delta tables.")
print("Status: PASS")

---

## Section 3: SQL Tests Against Materialized Tables

Verify that standard SQL operations work against the materialized Neo4j data.
These are the operations Genie needs to answer natural language questions.

In [ ]:
# =============================================================================
# TEST 1: GROUP BY — Products per company
# =============================================================================

print("TEST 1: GROUP BY — Products per company")
print("=" * 60)

spark.sql(f"""
    SELECT companyName, COUNT(*) AS product_count
    FROM {FQN}.neo4j_company_products
    GROUP BY companyName
    ORDER BY product_count DESC
""").show(truncate=False)

print("[PASS] GROUP BY on materialized Neo4j relationship data")

In [ ]:
# =============================================================================
# TEST 2: WHERE + ORDER BY — Filter risk factors by company
# =============================================================================

print("TEST 2: WHERE + ORDER BY — Microsoft risk factors")
print("=" * 60)

spark.sql(f"""
    SELECT riskId, riskName
    FROM {FQN}.neo4j_company_risks
    WHERE companyName = 'Microsoft Corporation'
    ORDER BY riskName
""").show(truncate=False)

print("[PASS] WHERE filter + ORDER BY on materialized Neo4j data")

In [ ]:
# =============================================================================
# TEST 3: Aggregations — Risk factors and competitors per company
# =============================================================================

print("TEST 3: Aggregations — Risk + competitor counts per company")
print("=" * 60)

spark.sql(f"""
    SELECT
        r.companyName,
        COUNT(DISTINCT r.riskId) AS risk_count,
        c.competitor_count
    FROM {FQN}.neo4j_company_risks r
    JOIN (
        SELECT companyId, COUNT(*) AS competitor_count
        FROM {FQN}.neo4j_competitors
        GROUP BY companyId
    ) c ON r.companyId = c.companyId
    GROUP BY r.companyName, c.competitor_count
    ORDER BY risk_count DESC
""").show(truncate=False)

print("[PASS] Aggregations with JOIN across materialized Neo4j tables")

In [ ]:
# =============================================================================
# TEST 4: DISTINCT — Unique competitors across all companies
# =============================================================================

print("TEST 4: DISTINCT — Unique competitors")
print("=" * 60)

spark.sql(f"""
    SELECT DISTINCT competitorName
    FROM {FQN}.neo4j_competitors
    WHERE competitorId IS NULL
    ORDER BY competitorName
""").show(50, truncate=False)

count = spark.sql(f"""
    SELECT COUNT(DISTINCT competitorName) AS cnt
    FROM {FQN}.neo4j_competitors
    WHERE competitorId IS NULL
""").collect()[0]["cnt"]
print(f"Found {count} unique external competitors")
print("[PASS] DISTINCT on materialized Neo4j relationship data")

---

## Section 4: Federated Queries — Materialized Tables + Delta Tables

The real power: JOIN materialized Neo4j tables with Delta lakehouse tables in standard SQL.
No JDBC, no Python driver, no Spark DataFrames — just SQL queries against Delta tables.

These are queries a **Genie agent** could answer via natural language.

In [ ]:
# =============================================================================
# FEDERATED QUERY 1: Company Overview
# =============================================================================
# "Give me a summary of each company: products, competitors, risks, and
#  financial metrics."
#
# Combines ALL data sources — materialized Neo4j + Delta lakehouse.

print("Federated Query 1: Company Overview Dashboard")
print("Neo4j materialized: products, competitors, risks")
print("Delta lakehouse: financial_metrics")
print("=" * 80)

spark.sql(f"""
    SELECT
        c.name AS company,
        c.ticker,
        p.product_count,
        comp.competitor_count,
        r.risk_count,
        fm.metric_count
    FROM {FQN}.companies c
    LEFT JOIN (
        SELECT companyId, COUNT(*) AS product_count
        FROM {FQN}.neo4j_company_products
        GROUP BY companyId
    ) p ON c.companyId = p.companyId
    LEFT JOIN (
        SELECT companyId, COUNT(*) AS competitor_count
        FROM {FQN}.neo4j_competitors
        GROUP BY companyId
    ) comp ON c.companyId = comp.companyId
    LEFT JOIN (
        SELECT companyId, COUNT(*) AS risk_count
        FROM {FQN}.neo4j_company_risks
        GROUP BY companyId
    ) r ON c.companyId = r.companyId
    LEFT JOIN (
        SELECT companyId, COUNT(*) AS metric_count
        FROM {FQN}.financial_metrics
        GROUP BY companyId
    ) fm ON c.companyId = fm.companyId
    ORDER BY c.companyId
""").show(truncate=False)

print("[PASS] Unified company dashboard from Neo4j + Delta")

In [ ]:
# =============================================================================
# FEDERATED QUERY 2: Competitor Exposure Analysis
# =============================================================================
# "Which asset managers hold shares in companies that compete with NVIDIA,
#  and how much are they exposed?"
#
# Graph traversal (competitors) + financial data (holdings) in pure SQL.

print("Federated Query 2: Competitor Exposure Analysis")
print("Neo4j materialized: neo4j_competitors")
print("Delta lakehouse: asset_manager_holdings, asset_managers")
print("=" * 80)

spark.sql(f"""
    SELECT
        am.name AS asset_manager,
        comp.competitorName AS nvidia_competitor,
        FORMAT_NUMBER(h.shares, 0) AS shares_held
    FROM {FQN}.neo4j_competitors comp
    JOIN {FQN}.asset_manager_holdings h
        ON comp.competitorId = h.companyId
    JOIN {FQN}.asset_managers am
        ON h.managerId = am.managerId
    WHERE comp.companyId = 'C004'
      AND comp.competitorId IS NOT NULL
    ORDER BY comp.competitorName, h.shares DESC
""").show(50, truncate=False)

print("[PASS] Graph traversal + financial holdings in pure SQL")

In [ ]:
# =============================================================================
# FEDERATED QUERY 3: Risk-Weighted Investment View
# =============================================================================
# "For each asset manager, how many risk factors do their portfolio companies
#  have, and what's their total share exposure?"
#
# The kind of question a risk analyst would ask a Genie agent.

print("Federated Query 3: Risk-Weighted Investment View")
print("Neo4j materialized: neo4j_company_risks")
print("Delta lakehouse: asset_manager_holdings, asset_managers, companies")
print("=" * 80)

spark.sql(f"""
    SELECT
        am.name AS asset_manager,
        COUNT(DISTINCT h.companyId) AS companies_held,
        FORMAT_NUMBER(SUM(h.shares), 0) AS total_shares,
        SUM(r.risk_count) AS total_risk_factors
    FROM {FQN}.asset_manager_holdings h
    JOIN {FQN}.asset_managers am ON h.managerId = am.managerId
    LEFT JOIN (
        SELECT companyId, COUNT(*) AS risk_count
        FROM {FQN}.neo4j_company_risks
        GROUP BY companyId
    ) r ON h.companyId = r.companyId
    GROUP BY am.name
    ORDER BY total_risk_factors DESC
""").show(truncate=False)

print("[PASS] Portfolio risk analysis from graph + lakehouse data")

In [ ]:
# =============================================================================
# FEDERATED QUERY 4: Product Portfolio + Financial Metrics
# =============================================================================
# "Show me each company's products alongside their key financial metrics."
#
# Joins graph data (products) with lakehouse data (financials) in one query.

print("Federated Query 4: Product Portfolio + Financial Metrics")
print("Neo4j materialized: neo4j_company_products")
print("Delta lakehouse: financial_metrics, companies")
print("=" * 80)

spark.sql(f"""
    WITH product_summary AS (
        SELECT companyId, CONCAT_WS(', ', COLLECT_LIST(productName)) AS products
        FROM {FQN}.neo4j_company_products
        GROUP BY companyId
    ),
    metric_sample AS (
        SELECT companyId,
               FIRST(name) AS sample_metric,
               FIRST(value) AS sample_value,
               FIRST(period) AS sample_period
        FROM {FQN}.financial_metrics
        GROUP BY companyId
    )
    SELECT
        c.name AS company,
        c.ticker,
        p.products,
        m.sample_metric,
        m.sample_value,
        m.sample_period
    FROM {FQN}.companies c
    JOIN product_summary p ON c.companyId = p.companyId
    JOIN metric_sample m ON c.companyId = m.companyId
    ORDER BY c.companyId
""").show(truncate=False)

print("[PASS] Product portfolio + financial metrics in pure SQL")

---

## Summary

This notebook materializes Neo4j graph data as **managed Delta tables** in Unity Catalog,
making it queryable alongside lakehouse tables with standard SQL.

### Materialized Tables

| Table | Source | Description |
|-------|--------|-------------|
| `neo4j_companies` | Company nodes | Companies with companyId, name, ticker |
| `neo4j_products` | Product nodes | Products with productId, name |
| `neo4j_risk_factors` | RiskFactor nodes | Risk factors with riskId, name |
| `neo4j_company_products` | OFFERS relationships | Company → Product mapping |
| `neo4j_competitors` | COMPETES_WITH relationships | Company → Competitor mapping |
| `neo4j_company_risks` | HAS_RISK relationships | Company → RiskFactor mapping |

### Materialization Approaches

| Approach | Used For | How |
|----------|----------|-----|
| `dbtable` + `customSchema` | Node labels (Company, Product, RiskFactor) | UC JDBC reads entire label as a table |
| Python driver → Spark DataFrame | Relationships (OFFERS, COMPETES_WITH, HAS_RISK) | Cypher query returns denormalized results |

### Next Steps

- **Add to a Genie space** — these tables can power natural language queries
- **Schedule refresh** — re-run Section 2 periodically to sync from Neo4j
- **Add more graph data** — materialize additional labels or relationship types as needed